In [53]:
import pandas as pd
import sqlite3
import os

conn = sqlite3.connect('../data/olist.db')

# 1. Carrega os dados com order_id
orders = pd.read_sql("""
    SELECT o.order_id, o.customer_id, c.customer_unique_id,
           o.order_purchase_timestamp, oi.price
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
""", conn)
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

# 2. RECÊNCIA
data_referencia = orders['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
recencia = orders.groupby('customer_unique_id')['order_purchase_timestamp'].max().reset_index()
recencia['recencia_dias'] = (data_referencia - recencia['order_purchase_timestamp']).dt.days
recencia = recencia[['customer_unique_id', 'recencia_dias']]

# 3. FREQUÊNCIA (corrigida: nunique de order_id)
frequencia = orders.groupby('customer_unique_id')['order_id'].nunique().reset_index()
frequencia.columns = ['customer_unique_id', 'frequencia']

# 4. MONETÁRIO
monetario = orders.groupby('customer_unique_id')['price'].sum().reset_index()
monetario.columns = ['customer_unique_id', 'monetario']

# 5. MERGE
rfm = recencia.merge(frequencia, on='customer_unique_id').merge(monetario, on='customer_unique_id')

# 6. SCORES
rfm['R'] = pd.qcut(rfm['recencia_dias'], q=5, labels=[5,4,3,2,1])
rfm['F'] = pd.qcut(rfm['frequencia'].rank(method='first'), q=5, labels=[1,2,3,4,5])
rfm['M'] = pd.qcut(rfm['monetario'], q=5, labels=[1,2,3,4,5])
rfm['rfm_score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)

# 7. SEGMENTAÇÃO
def segmentar(row):
    r, f, m = int(row['R']), int(row['F']), int(row['M'])
    if r >= 4 and f >= 4 and m >= 4:
        return 'Campeões'
    elif r >= 3 and f >= 3:
        return 'Clientes Fiéis'
    elif r >= 4 and f <= 2:
        return 'Novos Clientes'
    elif r <= 2 and f >= 3:
        return 'Em Risco'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Hibernando'
    else:
        return 'Potencial'

rfm['segmento'] = rfm.apply(segmentar, axis=1)

# 8. EXPORTA (já com sep=';' para o Power BI)
os.makedirs('../powerbi/dados', exist_ok=True)
rfm.to_csv('../powerbi/dados/rfm_clientes.csv', index=False, sep=';', decimal=',')

print(f"Clientes: {len(rfm):,}")
print(f"Recorrentes: {(rfm['frequencia']>=2).sum():,}")
print(rfm['segmento'].value_counts())

Clientes: 93,358
Recorrentes: 2,801
segmento
Clientes Fiéis    27336
Em Risco          22230
Potencial         15881
Novos Clientes    14984
Hibernando         6478
Campeões           6449
Name: count, dtype: int64
